In [0]:
%run ./Tools

In [0]:
guid = dbutils.widgets.get("guid")

In [0]:
bronze_characters_df = spark.read.table("bronze_characters").filter(f"guid == '{guid}'")
bronze_locations_df = spark.read.table("bronze_locations").filter(f"guid == '{guid}'")
bronze_episodes_df = spark.read.table("bronze_episodes").filter(f"guid == '{guid}'")

In [0]:
silver_characters_df = bronze_characters_df.select(
    F.col("created"),
    F.col("gender"),
    F.col("id").alias("character_id"),
    F.expr("extract_id(location.url)").try_cast(IntegerType()).alias("character_location"),
    F.col("name"),
    F.expr("extract_id(origin.url)").try_cast(IntegerType()).alias("character_origin"),
    F.col("species"),
    F.col("status"),
    F.col("type"),
    F.col("image"),
    F.col("guid")
)

saveToTableFromDelta(silver_characters_df, "silver_characters")

In [0]:
silver_locations_residents_df = bronze_locations_df.select(
    F.col("created"),
    F.col("dimension"),
    F.col("id").alias("location_id"),
    F.col("name"),
    F.explode(F.col("residents")).alias("resident_id"),
    F.col("type"),
    F.col("guid")
)
silver_locations_residents_df = silver_locations_residents_df.withColumn("resident_id", F.expr("extract_id(resident_id)").try_cast(IntegerType()))

dim_locations_df = bronze_locations_df.select(
    F.col("created"),
    F.col("dimension"),
    F.col("id").alias("location_id"),
    F.col("name"),
    F.col("type")
)

saveToTableFromDelta(silver_locations_residents_df, "silver_locations_residents")
saveToTableFromDelta(dim_locations_df, "silver_dim_locations")

In [0]:
silver_episodes_df = bronze_episodes_df.select(
    F.col("air_date"),
    F.explode(F.col("characters")).alias("character_id"),
    F.col("created"),
    F.col("episode"),
    F.col("id").alias("episode_id"),
    F.col("name"),
    F.col("guid")
)
silver_episodes_df = silver_episodes_df.withColumn("character_id", F.expr("extract_id(character_id)").try_cast(IntegerType()))

saveToTableFromDelta(silver_episodes_df, "silver_episodes")